# Flight Delay Prediction — Logistic Regression

**Input:** `train_sample.parquet` (stratified 10 % of train, seed=42) + `val.parquet`  
**Target:** `ArrDel15` — binary (1 = arrival ≥ 15 min late)  
**Prediction horizon:** T–2 hours before scheduled departure  

Pipeline: `VectorAssembler` → `StandardScaler` → `LogisticRegression`  
Evaluation: AUC-ROC · AUC-PR · F1 · Precision · Recall · Confusion matrix · Threshold tuning

> **Note:** Test set evaluation is intentionally deferred. Evaluate on test  
> only once a final model configuration is confirmed.

---
## 1. Environment Setup

In [1]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq
!pip install pyspark --quiet

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
import os, subprocess
result = subprocess.run(
    "java -XshowSettings:property -version 2>&1 | grep 'java.home'",
    shell=True, capture_output=True, text=True
)
os.environ['JAVA_HOME'] = result.stdout.strip().split('=')[-1].strip()
print('JAVA_HOME:', os.environ['JAVA_HOME'])

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('FlightDelay_LogReg')
    .master('local[*]')
    .config('spark.driver.memory', '6g')           # 10 % sample fits comfortably in 6 g
    .config('spark.sql.shuffle.partitions', '32')  # appropriate for ~3 M row sample
    .getOrCreate()
)
spark.conf.set('spark.sql.parquet.int96RebaseModeInRead', 'CORRECTED')
spark.conf.set('spark.sql.legacy.parquet.nanosAsLong', 'true')
spark.conf.set('spark.sql.parquet.mergeSchema', 'false')
spark.conf.set('spark.hadoop.parquet.enable.summary-metadata', 'false')
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Spark version: 4.0.2


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
### Uncomment and use in case of crash with Drive connection
#from google.colab import drive
#drive.flush_and_unmount()
#drive.mount('/content/drive', force_remount=True)

---
## 2. Load Data

Reads the pre-built 10 % training sample and the full validation set from Drive.  

Both are copied to local NVMe SSD before Spark reads them — Drive I/O is slow;  
all Spark operations then run off fast local storage.

No full training set is loaded. No test set is loaded.

Update `SAMPLE_PATH` if Drive folder differs or use the paired FlightDelay_DataPrep notebook to generate the sample.

In [6]:
from pyspark.sql import functions as F
import shutil, os

SAMPLE_PATH = '/content/drive/MyDrive/OMDS Capstone/Data/flights_sample'
LOCAL       = '/content/local_data'
os.makedirs(LOCAL, exist_ok=True)

for name in ['train_sample', 'val']:
    shutil.copytree(
        f'{SAMPLE_PATH}/{name}.parquet',
        f'{LOCAL}/{name}.parquet',
        dirs_exist_ok=True
    )
    print(f'Copied {name} to local SSD')

train_df = spark.read.parquet(f'{LOCAL}/train_sample.parquet')
val_df   = spark.read.parquet(f'{LOCAL}/val.parquet')

print(f'\nTrain sample rows : {train_df.count():>10,}')
print(f'Val rows          : {val_df.count():>10,}')

Copied train_sample to local SSD
Copied val to local SSD

Train sample rows :  3,115,085
Val rows          :  6,743,403


---
## 3. Feature Definition

Canonical feature list — must match `FlightDelay_DataPrep.ipynb` exactly. Some features may be dropped if multicollinarity or memory constraints become an issue

| Group | Features |
|---|---|
| Schedule / calendar | `Month`, `DayOfWeek`, `dep_hour`, `Distance`, `CRSElapsedTime`, `is_weekend`, `is_holiday` |
| Carrier rolling | `carrier_delay_rate_30d`, `carrier_delay_rate_90d` |
| Airport congestion | `origin_departures_3h` |
| Target encoded | `origin_delay_rate`, `dest_delay_rate` |
| Weather flags (origin) | `origin_is_rain`, `origin_is_snow`, `origin_is_fog`, `origin_low_visibility`, `origin_high_wind`, `origin_severe_weather` |
| Weather flags (dest) | `dest_is_rain`, `dest_is_snow`, `dest_is_fog`, `dest_low_visibility`, `dest_high_wind`, `dest_severe_weather` |
| Weather continuous (origin) | `origin_visibility`, `origin_wind_kts`, `origin_gust_kts`, `origin_precip_in` |
| Weather continuous (dest) | `dest_visibility`, `dest_wind_kts`, `dest_gust_kts`, `dest_precip_in` |

In [7]:
SCHEDULE_FEATURES = [
    'Month', 'DayOfWeek', 'dep_hour', 'Distance', 'CRSElapsedTime',
    'is_holiday',
    # dropped: is_weekend (derivable from DayOfWeek)
]
ENGINEERED_FEATURES = [
    'carrier_delay_rate_30d',
    # dropped: carrier_delay_rate_90d (collinear with 30d)
    'origin_departures_3h',
    'origin_delay_rate', 'dest_delay_rate',
]
WEATHER_FLAGS = [
    'origin_is_rain', 'origin_is_snow', 'origin_is_fog', 'origin_severe_weather',
    'dest_is_rain',   'dest_is_snow',   'dest_is_fog',   'dest_severe_weather',
    # dropped: origin/dest_low_visibility (captured by continuous visibility)
    # dropped: origin/dest_high_wind (captured by continuous wind_kts)
]
WEATHER_CONTINUOUS = [
    'origin_visibility', 'origin_wind_kts', 'origin_precip_in',
    'dest_visibility',   'dest_wind_kts',   'dest_precip_in',
    # dropped: origin/dest_gust_kts (collinear with wind_kts)
]
FEATURE_COLS = SCHEDULE_FEATURES + ENGINEERED_FEATURES + WEATHER_FLAGS + WEATHER_CONTINUOUS
TARGET_COL   = 'ArrDel15'
WEIGHT_COL   = 'class_weight'

print(f'Total features : {len(FEATURE_COLS)}')

Total features : 24


---
## 4. Preprocessing Pipeline — Definition

`VectorAssembler` stacks features into a dense vector.  
`StandardScaler` (mean=0, std=1) —
The scaler is fit on the training sample

`standardization=False` is set on `LogisticRegression` to prevent double-scaling.

In [8]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression

assembler = VectorAssembler(
    inputCols=FEATURE_COLS,
    outputCol='raw_features',
    handleInvalid='skip'
)
scaler = StandardScaler(
    inputCol='raw_features',
    outputCol='features',
    withMean=True,
    withStd=True
)
prep_pipeline = Pipeline(stages=[assembler, scaler])
print('Pipeline objects defined (not yet fitted).')
print(f'Feature vector length: {len(FEATURE_COLS)}')

Pipeline objects defined (not yet fitted).
Feature vector length: 24


---
## 5. Evaluation Helper

Centralised function used throughout §§6–8 so metrics are computed consistently.

In [9]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

def evaluate_predictions(predictions, label_col=TARGET_COL, threshold=0.5, split_name=''):
    """Compute AUC-ROC, AUC-PR, F1, Precision, Recall, Accuracy, and confusion matrix."""
    auc_roc = BinaryClassificationEvaluator(
        labelCol=label_col, rawPredictionCol='rawPrediction', metricName='areaUnderROC'
    ).evaluate(predictions)
    auc_pr = BinaryClassificationEvaluator(
        labelCol=label_col, rawPredictionCol='rawPrediction', metricName='areaUnderPR'
    ).evaluate(predictions)

    preds_t = predictions.withColumn(
        'pred_label',
        (vector_to_array(F.col('probability'))[1] >= threshold).cast('double')
    )
    mc        = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol='pred_label')
    f1        = mc.setMetricName('f1').evaluate(preds_t)
    precision = mc.setMetricName('weightedPrecision').evaluate(preds_t)
    recall    = mc.setMetricName('weightedRecall').evaluate(preds_t)
    accuracy  = mc.setMetricName('accuracy').evaluate(preds_t)

    cm = (
        preds_t
        .groupBy(F.col(label_col).cast('int').alias('actual'),
                 F.col('pred_label').cast('int').alias('predicted'))
        .count().orderBy('actual', 'predicted')
    )
    if split_name:
        print(f'\n=== {split_name} ===')
    print(f'  AUC-ROC   : {auc_roc:.4f}')
    print(f'  AUC-PR    : {auc_pr:.4f}')
    print(f'  Accuracy  : {accuracy:.4f}')
    print(f'  F1        : {f1:.4f}')
    print(f'  Precision : {precision:.4f}')
    print(f'  Recall    : {recall:.4f}')
    print('  Confusion matrix (rows=actual, cols=predicted):')
    cm.show()
    return dict(auc_roc=auc_roc, auc_pr=auc_pr, f1=f1,
                precision=precision, recall=recall, accuracy=accuracy)

---
## 6. Prepare Training & Validation Data

Steps:
1. Compute class imbalance ratio from the training sample and add a `class_weight` column
2. Impute weather nulls to 0 (calm/clear baseline) on both splits
3. Fit the preprocessing pipeline on the training sample
4. Materialise `train_prep` (checkpoint → cache) to break Spark lineage
5. Write `val_prep` to local SSD — reloaded fresh per model cell to avoid  
   holding two large feature matrices in RAM simultaneously (Recommended by Claude and seemed to help)

In [10]:
# Claude was used to help with figuring out checkpointing and caching logic to try and optimize to manage Google Colab (Free) Constraints
CKPT_DIR = '/content/local_data/checkpoints'
VAL_PREP_PATH = '/content/local_data/val_prep.parquet'
os.makedirs(CKPT_DIR, exist_ok=True)
spark.sparkContext.setCheckpointDir(CKPT_DIR)
print('✓ Checkpoint dir set.')

# Class weights
print('Computing class counts...')
counts_map      = {r[TARGET_COL]: r['count'] for r in train_df.groupBy(TARGET_COL).count().collect()}
imbalance_ratio = counts_map[0] / counts_map[1]
print(f'Not delayed : {counts_map[0]:>10,}')
print(f'Delayed     : {counts_map[1]:>10,}')
print(f'Ratio       : {imbalance_ratio:.2f}:1')

print('Applying sample weights column...')
train_df = train_df.withColumn(
    WEIGHT_COL,
    F.when(F.col(TARGET_COL) == 1, imbalance_ratio).otherwise(F.lit(1.0))
)
print(f'✓ Weight column "{WEIGHT_COL}" added.')

# Impute weather nulls
print('Filling weather nulls...')
fill_map = {col: 0 for col in WEATHER_FLAGS + WEATHER_CONTINUOUS}
train_df = train_df.fillna(fill_map)
val_df   = val_df.fillna(fill_map)
print(f'✓ Filled {len(fill_map)} columns with 0.')

# Fit pipeline on training sample
print('Fitting prep pipeline (this triggers a Spark job)...')
prep_model = prep_pipeline.fit(train_df)
print('✓ prep_pipeline fitted.')

# train_prep: checkpoint → cache → materialise
print('Transforming train_df...')
train_prep_raw = prep_model.transform(train_df)
print('✓ Transform defined (lazy).')

print('Checkpointing train_prep (triggers Spark job + disk write)...')
train_prep = train_prep_raw.checkpoint()
print('✓ Checkpoint complete.')

print('Caching train_prep...')
train_prep.cache()
print('✓ Cache registered (lazy).')

print('Counting train_prep to materialise (triggers Spark job)...')
n = train_prep.count()
print(f'✓ train_prep materialised: {n:,} rows.')

train_df.unpersist()
print('✓ train_df unpersisted.')

# val_prep: write to SSD, reload fresh per model cell
print('Transforming val_df...')
val_prep_raw = prep_model.transform(val_df)
print('✓ val_df transform defined (lazy).')

print(f'Writing val_prep to {VAL_PREP_PATH}...')
val_prep_raw.coalesce(8).write.mode('overwrite').parquet(VAL_PREP_PATH)
print('✓ val_prep written to local SSD.')

val_df.unpersist()
print('✓ val_df unpersisted.')

✓ Checkpoint dir set.
Computing class counts...
Not delayed :  2,558,185
Delayed     :    556,900
Ratio       : 4.59:1
Applying sample weights column...
✓ Weight column "class_weight" added.
Filling weather nulls...
✓ Filled 14 columns with 0.
Fitting prep pipeline (this triggers a Spark job)...
✓ prep_pipeline fitted.
Transforming train_df...
✓ Transform defined (lazy).
Checkpointing train_prep (triggers Spark job + disk write)...
✓ Checkpoint complete.
Caching train_prep...
✓ Cache registered (lazy).
Counting train_prep to materialise (triggers Spark job)...
✓ train_prep materialised: 3,115,085 rows.
✓ train_df unpersisted.
Transforming val_df...
✓ val_df transform defined (lazy).
Writing val_prep to /content/local_data/val_prep.parquet...
✓ val_prep written to local SSD.
✓ val_df unpersisted.


---
## 7. Regularisation Sweep

Sweeps `regParam` over a log-scale grid. Model selection is based on  
**validation AUC-ROC** (threshold-independent, so model selection and  
threshold tuning remain separate decisions).  
Pure L2 (`elasticNetParam=0.0`) throughout — the feature set is small  
and dense so L1 sparsity is not needed.

Run each model cell individually. `val_prep` is reloaded from SSD before  
each fit to avoid holding two large feature matrices in RAM at once.

In [11]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_eval   = BinaryClassificationEvaluator(
    labelCol=TARGET_COL, rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)
aucpr_eval = BinaryClassificationEvaluator(
    labelCol=TARGET_COL, rawPredictionCol='rawPrediction', metricName='areaUnderPR'
)
sweep_results = []
print('Evaluators ready. Run each model cell below.')

Evaluators ready. Run each model cell below.


In [12]:
# Model A — regParam = 0.01
val_prep = spark.read.parquet(VAL_PREP_PATH)
lr_A = LogisticRegression(
    featuresCol='features', labelCol=TARGET_COL, weightCol=WEIGHT_COL,
    regParam=0.01, elasticNetParam=0.0, maxIter=50, standardization=False
)
m_A     = lr_A.fit(train_prep)
preds_A = m_A.transform(val_prep)
auc_A   = auc_eval.evaluate(preds_A)
aucpr_A = aucpr_eval.evaluate(preds_A)
preds_A.unpersist()
sweep_results.append(('A', 0.01, auc_A, aucpr_A, m_A.summary.totalIterations, m_A))
print(f'A  regParam=0.01  AUC-ROC={auc_A:.4f}  AUC-PR={aucpr_A:.4f}  iters={m_A.summary.totalIterations}')

A  regParam=0.01  AUC-ROC=0.6765  AUC-PR=0.3533  iters=20


In [13]:
# Model B — regParam = 0.1
val_prep = spark.read.parquet(VAL_PREP_PATH)
lr_B = LogisticRegression(
    featuresCol='features', labelCol=TARGET_COL, weightCol=WEIGHT_COL,
    regParam=0.1, elasticNetParam=0.0, maxIter=50, standardization=False
)
m_B     = lr_B.fit(train_prep)
preds_B = m_B.transform(val_prep)
auc_B   = auc_eval.evaluate(preds_B)
aucpr_B = aucpr_eval.evaluate(preds_B)
preds_B.unpersist()
sweep_results.append(('B', 0.1, auc_B, aucpr_B, m_B.summary.totalIterations, m_B))
print(f'B  regParam=0.1   AUC-ROC={auc_B:.4f}  AUC-PR={aucpr_B:.4f}  iters={m_B.summary.totalIterations}')

B  regParam=0.1   AUC-ROC=0.6757  AUC-PR=0.3525  iters=14


In [14]:
# Model C — regParam = 0.5
val_prep = spark.read.parquet(VAL_PREP_PATH)
lr_C = LogisticRegression(
    featuresCol='features', labelCol=TARGET_COL, weightCol=WEIGHT_COL,
    regParam=0.5, elasticNetParam=0.0, maxIter=50, standardization=False
)
m_C     = lr_C.fit(train_prep)
preds_C = m_C.transform(val_prep)
auc_C   = auc_eval.evaluate(preds_C)
aucpr_C = aucpr_eval.evaluate(preds_C)
preds_C.unpersist()
sweep_results.append(('C', 0.5, auc_C, aucpr_C, m_C.summary.totalIterations, m_C))
print(f'C  regParam=0.5   AUC-ROC={auc_C:.4f}  AUC-PR={aucpr_C:.4f}  iters={m_C.summary.totalIterations}')

C  regParam=0.5   AUC-ROC=0.6709  AUC-PR=0.3459  iters=10


In [15]:
best = max(sweep_results, key=lambda x: x[2])  # by AUC-ROC
label, best_reg, best_auc, best_aucpr, best_iters, best_model = best

print('Sweep summary:')
print(f"{'Model':<8} {'regParam':>10} {'AUC-ROC':>10} {'AUC-PR':>10} {'iters':>8}")
print('-' * 50)
for row in sweep_results:
    marker = ' <-- best' if row[0] == label else ''
    print(f"{row[0]:<8} {row[1]:>10.3f} {row[2]:>10.4f} {row[3]:>10.4f} {row[4]:>8}{marker}")
print(f'\nBest: Model {label}  regParam={best_reg}  AUC-ROC={best_auc:.4f}')

Sweep summary:
Model      regParam    AUC-ROC     AUC-PR    iters
--------------------------------------------------
A             0.010     0.6765     0.3533       20 <-- best
B             0.100     0.6757     0.3525       14
C             0.500     0.6709     0.3459       10

Best: Model A  regParam=0.01  AUC-ROC=0.6765


In [16]:
# Coefficient table — Used to inspect direction and magnitude of each feature
import pandas as pd

coef_df = (
    pd.DataFrame({'feature': FEATURE_COLS, 'coefficient': best_model.coefficients.toArray()})
    .assign(abs_coef=lambda d: d.coefficient.abs())
    .sort_values('abs_coef', ascending=False)
    .drop(columns='abs_coef')
)
print('Feature coefficients — best model (sorted by |coef|):')
print(coef_df.to_string(index=False))

Feature coefficients — best model (sorted by |coef|):
               feature  coefficient
carrier_delay_rate_30d     0.386187
              dep_hour     0.307745
        origin_is_snow     0.128219
         dest_wind_kts     0.091749
          dest_is_rain     0.077181
                 Month     0.076828
       dest_delay_rate     0.073442
       dest_visibility    -0.071976
 origin_severe_weather     0.070145
       origin_wind_kts     0.063441
        origin_is_rain     0.063025
     origin_delay_rate     0.062600
   dest_severe_weather     0.062437
     origin_visibility    -0.057966
              Distance     0.057216
  origin_departures_3h     0.046889
      origin_precip_in     0.046365
          dest_is_snow     0.045416
        dest_precip_in     0.038876
             DayOfWeek     0.022131
         origin_is_fog     0.020126
        CRSElapsedTime    -0.018323
            is_holiday    -0.011318
           dest_is_fog     0.009137


---
## 8. Threshold Tuning on Validation Set

The default threshold of 0.5 assumes balanced classes. With ~20 % positives,  
a lower threshold recovers recall at a modest precision cost. Here, a missed delay is more costly than a false alarm.  

Threshold is swept on the **validation set only**. The optimal value is recorded in model metadata for use when test evaluation is eventually run.

In [17]:
val_prep          = spark.read.parquet(VAL_PREP_PATH)
val_preds_best    = best_model.transform(val_prep)
val_with_prob     = val_preds_best.withColumn(
    'prob_pos', vector_to_array(F.col('probability'))[1]
).cache()

mc_f1     = MulticlassClassificationEvaluator(labelCol=TARGET_COL, predictionCol='thresh_pred', metricName='f1')
mc_prec   = MulticlassClassificationEvaluator(labelCol=TARGET_COL, predictionCol='thresh_pred', metricName='weightedPrecision')
mc_recall = MulticlassClassificationEvaluator(labelCol=TARGET_COL, predictionCol='thresh_pred', metricName='weightedRecall')

THRESHOLDS       = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
threshold_results = []

print(f"{'Threshold':>10} {'F1':>8} {'Precision':>10} {'Recall':>8}")
print('-' * 42)
for t in THRESHOLDS:
    preds_t = val_with_prob.withColumn('thresh_pred', (F.col('prob_pos') >= t).cast('double'))
    f1      = mc_f1.evaluate(preds_t)
    prec    = mc_prec.evaluate(preds_t)
    rec     = mc_recall.evaluate(preds_t)
    threshold_results.append((t, f1, prec, rec))
    print(f'{t:>10.2f} {f1:>8.4f} {prec:>10.4f} {rec:>8.4f}')

best_threshold = max(threshold_results, key=lambda x: x[1])[0]
print(f'\nOptimal threshold : {best_threshold}')

 Threshold       F1  Precision   Recall
------------------------------------------
      0.20   0.0762     0.7912   0.2084
      0.25   0.1133     0.7859   0.2259
      0.30   0.1972     0.7799   0.2682
      0.35   0.3124     0.7742   0.3342
      0.40   0.4330     0.7680   0.4159
      0.45   0.5401     0.7611   0.5029
      0.50   0.6267     0.7549   0.5873
      0.55   0.6881     0.7491   0.6590
      0.60   0.7272     0.7443   0.7147

Optimal threshold : 0.6


In [18]:
val_metrics = evaluate_predictions(
    val_preds_best,
    threshold=best_threshold,
    split_name=f'Validation — regParam={best_reg}, threshold={best_threshold}'
)


=== Validation — regParam=0.01, threshold=0.6 ===
  AUC-ROC   : 0.6765
  AUC-PR    : 0.3533
  Accuracy  : 0.7147
  F1        : 0.7272
  Precision : 0.7443
  Recall    : 0.7147
  Confusion matrix (rows=actual, cols=predicted):
+------+---------+-------+
|actual|predicted|  count|
+------+---------+-------+
|     0|        0|4193741|
|     0|        1|1162963|
|     1|        0| 760905|
|     1|        1| 625794|
+------+---------+-------+



---
## 9. Save Model & Metadata

Saves the preprocessing pipeline, the best logistic regression model,  
and a JSON metadata file.  
The preprocessing pipeline is saved separately so it can be reused by  
subsequent model notebooks (random forest, XGBoost) without refitting.

> **Test metrics are intentionally absent** from metadata at this stage.  
> A `test_auc_roc: null` placeholder is included as a reminder.

In [19]:
import json

MODEL_BASE = '/content/drive/MyDrive/OMDS Capstone/Models/logreg'

prep_model.write().overwrite().save(f'{MODEL_BASE}/prep_pipeline')
print('Saved: prep_pipeline')

best_model.write().overwrite().save(f'{MODEL_BASE}/lr_model')
print('Saved: lr_model')

metadata = {
    'model_type'        : 'LogisticRegression',
    'training_data'     : 'train_sample.parquet (stratified 10%, seed=42)',
    'reg_param'         : best_reg,
    'elastic_net_param' : 0.0,
    'max_iter'          : 50,
    'optimal_threshold' : best_threshold,
    'feature_cols'      : FEATURE_COLS,
    'target_col'        : TARGET_COL,
    'val_auc_roc'       : round(val_metrics['auc_roc'],   4),
    'val_auc_pr'        : round(val_metrics['auc_pr'],    4),
    'val_f1'            : round(val_metrics['f1'],        4),
    'val_precision'     : round(val_metrics['precision'], 4),
    'val_recall'        : round(val_metrics['recall'],    4),
    # Test metrics deferred
    'test_auc_roc'      : None,
    'test_auc_pr'       : None,
    'test_f1'           : None,
}

with open(f'{MODEL_BASE}/metadata.json', 'w') as fh:
    json.dump(metadata, fh, indent=2)
print('Saved: metadata.json')
print()
print(json.dumps(metadata, indent=2))

Saved: prep_pipeline
Saved: lr_model
Saved: metadata.json

{
  "model_type": "LogisticRegression",
  "training_data": "train_sample.parquet (stratified 10%, seed=42)",
  "reg_param": 0.01,
  "elastic_net_param": 0.0,
  "max_iter": 50,
  "optimal_threshold": 0.6,
  "feature_cols": [
    "Month",
    "DayOfWeek",
    "dep_hour",
    "Distance",
    "CRSElapsedTime",
    "is_holiday",
    "carrier_delay_rate_30d",
    "origin_departures_3h",
    "origin_delay_rate",
    "dest_delay_rate",
    "origin_is_rain",
    "origin_is_snow",
    "origin_is_fog",
    "origin_severe_weather",
    "dest_is_rain",
    "dest_is_snow",
    "dest_is_fog",
    "dest_severe_weather",
    "origin_visibility",
    "origin_wind_kts",
    "origin_precip_in",
    "dest_visibility",
    "dest_wind_kts",
    "dest_precip_in"
  ],
  "target_col": "ArrDel15",
  "val_auc_roc": 0.6765,
  "val_auc_pr": 0.3533,
  "val_f1": 0.7272,
  "val_precision": 0.7443,
  "val_recall": 0.7147,
  "test_auc_roc": null,
  "test_auc_pr"